# DipRadar v3 — Treino com Momentum Features + Comparação de Modelos

**Objetivo**: Comparar modelos v2 (19 features) com v3 (23 features = v2 + Stage 3b momentum).  
Escolher o *champion* com base em walk-forward Spearman + Top-K P&L.

## O que mudou face ao notebook anterior

| | v2 (anterior) | v3 (este notebook) |
|---|---|---|
| Features | 19 | **23** (+ `return_1m`, `return_3m_pre`, `sector_relative`, `beta_60d`) |
| Targets | `max_return_60d`, `max_drawdown_60d` | idem |
| Modelos | XGBoost up + down | **XGB + LightGBM + RandomForest** (6 modelos) |
| Validação | single split | **Walk-Forward** (5 folds, expanding window) |
| Seleção | manual | **champion automático** por Spearman rank + Top-K P&L |

## Arquitetura do pipeline v3

```
alert_db.csv  →  build_dataset_v3()  →  dataset_v3.parquet
                 (momentum features via yfinance + SPY/sector ETF)
                      ↓
            walk_forward_cv()  →  metricas por fold
                      ↓
         champion_select()  →  dip_models_v3.pkl
```


## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

# Ajustar path consoante ambiente (Colab vs local)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'ml_features.py').exists():
    REPO_ROOT = REPO_ROOT.parent
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Não encontrei ml_features.py — executa este notebook a partir da raiz do repo')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'REPO_ROOT: {REPO_ROOT}')

# Instalar dependências se necessário
import subprocess
for pkg in ['xgboost', 'lightgbm', 'scikit-learn', 'yfinance', 'pyarrow']:    
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Dependências OK')


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import logging
import pickle
import time
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf

from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb

from ml_features import (
    FEATURE_COLUMNS, N_FEATURES,
    add_derived_features, add_momentum_features, _FALLBACK,
)
from experiments.ml_v2.pipeline import (
    FEATURE_COLUMNS_V2, build_targets, build_v2_features,
    transform_targets, inverse_transform,
)

logging.basicConfig(level=logging.WARNING)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────
DATA_DIR    = REPO_ROOT
ALERT_DB    = DATA_DIR / 'alert_db.csv'
PARQUET_V1  = DATA_DIR / 'ml_training_merged.parquet'   # dataset v1/v2
PARQUET_V3  = REPO_ROOT / 'experiments/ml_v2/dataset_v3.parquet'  # output deste notebook
MODELS_DIR  = REPO_ROOT / 'experiments/ml_v2'

print(f'ALERT_DB   exists: {ALERT_DB.exists()}')
print(f'PARQUET_V1 exists: {PARQUET_V1.exists()}')
print(f'PARQUET_V3 exists (ainda nao): {PARQUET_V3.exists()}')


## 1. Carregar dataset v1/v2 (baseline)

In [ ]:
# Dataset v1/v2 — 19 features, base de comparação
df_v1 = pd.read_parquet(PARQUET_V1)

# Verificar colunas disponíveis
print(f'Shape: {df_v1.shape}')
print(f'Colunas: {list(df_v1.columns)}')
print()

# Identificar colunas de meta, features e labels
LABEL_WIN_COL   = 'label_win'
LABEL_DROP_COL  = 'label_further_drop'

v1_features = [c for c in FEATURE_COLUMNS if c in df_v1.columns]
v2_extra    = [c for c in FEATURE_COLUMNS_V2 if c not in FEATURE_COLUMNS and c in df_v1.columns]

print(f'Features v1 disponíveis no parquet: {len(v1_features)}/{len(FEATURE_COLUMNS)}')
print(f'Features v2 extra disponíveis     : {len(v2_extra)}')
print()

# Distribuição do target
if LABEL_WIN_COL in df_v1.columns:
    win_rate = df_v1[LABEL_WIN_COL].mean()
    print(f'Win rate (label_win=1): {win_rate:.1%}')
if LABEL_DROP_COL in df_v1.columns:
    print(f'label_further_drop: mean={df_v1[LABEL_DROP_COL].mean():.3f}  std={df_v1[LABEL_DROP_COL].std():.3f}')


## 2. Enriquecer dataset com features de Momentum (Stage 3b)

### 2a. Fetch de preços históricos (SPY + sector ETFs + stocks)

Esta célula faz download dos preços necessários para calcular:
- `return_1m`, `return_3m_pre` — preços do stock 21/63 dias antes do alerta
- `sector_relative` — idem para o ETF do sector
- `beta_60d` — OLS beta vs SPY nos 60 dias antes do alerta

**Tempo estimado**: ~15-25 min para o dataset completo (1 request yfinance por ticker).
Usa `MAX_TICKERS = None` para processar todos, ou um número para testar.

In [ ]:
# Mapeamento sector → ETF (espelha macro_data.py)
SECTOR_ETF = {
    'Technology': 'XLK', 'Healthcare': 'XLV',
    'Communication Services': 'XLC', 'Financial Services': 'XLF',
    'Financials': 'XLF', 'Consumer Cyclical': 'XLY',
    'Consumer Defensive': 'XLP', 'Industrials': 'XLI',
    'Energy': 'XLE', 'Utilities': 'XLU',
    'Real Estate': 'XLRE', 'Basic Materials': 'XLB',
    'Materials': 'XLB',
}
DEFAULT_ETF = 'SPY'

# ── Parâmetros ──────────────────────────────────────────────────────────
MAX_TICKERS  = None   # None = todos; int = limitar para testes (ex: 20)
HISTORY_DAYS = 130    # dias de historial antes do alerta mais antigo
HORIZON_DAYS = 65     # dias de futuro após o alerta mais recente
YF_SLEEP     = 0.35   # segundos entre requests

# Identificar colunas de data e ticker no parquet
date_col   = 'alert_date' if 'alert_date' in df_v1.columns else 'date'
ticker_col = 'ticker'    if 'ticker'     in df_v1.columns else 'symbol'
price_col  = 'entry_price' if 'entry_price' in df_v1.columns else 'price'
sector_col = 'sector'    if 'sector'     in df_v1.columns else None

df_work = df_v1.copy()
df_work['alert_date'] = pd.to_datetime(df_work[date_col], errors='coerce')
df_work['ticker']     = df_work[ticker_col].str.strip().str.upper()
df_work['entry_price'] = pd.to_numeric(df_work.get(price_col, pd.Series(dtype=float)), errors='coerce')
if sector_col:
    df_work['sector'] = df_work[sector_col].fillna('Unknown')
else:
    df_work['sector'] = 'Unknown'

df_work = df_work.dropna(subset=['alert_date', 'ticker'])

unique_tickers = df_work['ticker'].unique()
if MAX_TICKERS:
    unique_tickers = unique_tickers[:MAX_TICKERS]
    df_work = df_work[df_work['ticker'].isin(unique_tickers)]

# Determinar ETFs únicos necessários
unique_sectors = df_work['sector'].unique()
etfs_needed = list(set(
    [SECTOR_ETF.get(s, DEFAULT_ETF) for s in unique_sectors] + ['SPY']
))

print(f'Tickers a processar : {len(unique_tickers)}')
print(f'ETFs a fazer fetch  : {etfs_needed}')
print(f'Alertas na janela   : {len(df_work)}')


In [ ]:
# ── Fetch de preços (1 request por ticker, batch) ─────────────────────
# Cobertura: [earliest_alert - HISTORY_DAYS, latest_alert + HORIZON_DAYS]

price_cache: dict[str, pd.DataFrame] = {}

def _fetch(ticker: str, start: str, end: str) -> pd.DataFrame | None:
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        return df if not df.empty else None
    except Exception as e:
        print(f'  ERRO {ticker}: {e}')
        return None

# 1. Fetch dos ETFs de sector + SPY (cobrindo todo o periodo do dataset)
global_start = (df_work['alert_date'].min() - pd.Timedelta(days=HISTORY_DAYS)).strftime('%Y-%m-%d')
global_end   = (df_work['alert_date'].max() + pd.Timedelta(days=HORIZON_DAYS + 5)).strftime('%Y-%m-%d')

print(f'Janela global: {global_start} → {global_end}')
print()

etf_cache: dict[str, pd.DataFrame] = {}
for etf in etfs_needed:
    print(f'  Fetch ETF {etf}...')
    etf_cache[etf] = _fetch(etf, global_start, global_end)
    time.sleep(YF_SLEEP)

print(f'ETFs carregados: {sum(v is not None for v in etf_cache.values())}/{len(etf_cache)}')
print()

# 2. Fetch dos stocks
ticker_groups = df_work.groupby('ticker')
n_total = len(unique_tickers)

for i, ticker in enumerate(unique_tickers, 1):
    group = ticker_groups.get_group(ticker)
    t_start = (group['alert_date'].min() - pd.Timedelta(days=HISTORY_DAYS)).strftime('%Y-%m-%d')
    t_end   = (group['alert_date'].max() + pd.Timedelta(days=HORIZON_DAYS + 5)).strftime('%Y-%m-%d')
    if i % 20 == 0:
        print(f'  [{i}/{n_total}] {ticker}...')
    price_cache[ticker] = _fetch(ticker, t_start, t_end)
    time.sleep(YF_SLEEP)

ok = sum(v is not None for v in price_cache.values())
print(f'\nStocks com precos: {ok}/{n_total}')


### 2b. Construir dataset_v3 (v2 features + momentum)

In [ ]:
# ── Novas features de momentum (Stage 3b) ─────────────────────────────
MOMENTUM_FEATURES = ['return_1m', 'return_3m_pre', 'sector_relative', 'beta_60d']

# FEATURE_COLUMNS_V3 = v2 + 4 momentum features
# (FEATURE_COLUMNS já tem as 5 engineered features adicionadas no commit de hoje)
FEATURE_COLUMNS_V3 = FEATURE_COLUMNS_V2 + MOMENTUM_FEATURES
print(f'Features v2: {len(FEATURE_COLUMNS_V2)}')
print(f'Features v3: {len(FEATURE_COLUMNS_V3)} (+ {len(MOMENTUM_FEATURES)} momentum)')
print()

rows_v3 = []
skipped = 0

for _, row in df_work.iterrows():
    ticker     = row['ticker']
    alert_date = pd.Timestamp(row['alert_date'])
    sector     = row.get('sector', 'Unknown')
    etf        = SECTOR_ETF.get(sector, DEFAULT_ETF)

    ohlcv = price_cache.get(ticker)
    if ohlcv is None:
        skipped += 1
        continue

    # Precos apenas ate alert_date (anti-leakage)
    hist = ohlcv[ohlcv.index <= alert_date].copy()
    if len(hist) < 25:
        skipped += 1
        continue

    # Features v1 (do parquet — point-in-time)
    fv = {}
    for col in FEATURE_COLUMNS:
        if col in row.index:
            try: fv[col] = float(row[col])
            except: fv[col] = _FALLBACK.get(col, 0.0)
        else:
            fv[col] = _FALLBACK.get(col, 0.0)
    add_derived_features(fv)  # recalcula as 5 engineered para consistencia

    # Features v2 extra (da price_history)
    fv2 = build_v2_features(row, hist)
    fv.update(fv2)

    # Features v3: momentum (Stage 3b)
    spy_hist    = etf_cache.get('SPY')
    sector_hist = etf_cache.get(etf)

    if spy_hist is not None:
        spy_slice = spy_hist[spy_hist.index <= alert_date]
    else:
        spy_slice = None
    if sector_hist is not None:
        sec_slice = sector_hist[sector_hist.index <= alert_date]
    else:
        sec_slice = None

    add_momentum_features(fv, hist, sec_slice, spy_slice)

    # Recuperar targets do parquet ou recalcular
    if 'max_return_60d' in row.index and not pd.isna(row.get('max_return_60d')):
        max_ret  = float(row['max_return_60d'])
        max_draw = float(row.get('max_drawdown_60d', 0.0))
    else:
        # Recalcular targets a partir dos precos
        entry_price = float(row.get('entry_price', row.get('price', 0.0)))
        if entry_price <= 0:
            skipped += 1
            continue
        future = ohlcv[
            (ohlcv.index > alert_date) &
            (ohlcv.index <= alert_date + pd.Timedelta(days=HORIZON_DAYS))
        ]['Close']
        if len(future) < 5:
            skipped += 1
            continue
        tgt = build_targets(alert_date, entry_price, future)
        if np.isnan(tgt['max_return_60d']):
            skipped += 1
            continue
        max_ret  = tgt['max_return_60d']
        max_draw = tgt['max_drawdown_60d']

    record = {
        'ticker':     ticker,
        'alert_date': alert_date,
        'sector':     sector,
        **{c: fv[c] for c in FEATURE_COLUMNS_V3 if c in fv},
        'max_return_60d':   max_ret,
        'max_drawdown_60d': max_draw,
        'label_win':        int(row.get('label_win', 0)),
    }
    rows_v3.append(record)

df_v3 = pd.DataFrame(rows_v3)
print(f'Dataset v3 shape: {df_v3.shape}')
print(f'Amostras saltadas: {skipped}')
print(f'Win rate: {df_v3["label_win"].mean():.1%}')
print(f'max_return_60d : mean={df_v3["max_return_60d"].mean():.4f}  std={df_v3["max_return_60d"].std():.4f}')
print(f'max_drawdown_60d: mean={df_v3["max_drawdown_60d"].mean():.4f}  std={df_v3["max_drawdown_60d"].std():.4f}')


In [ ]:
# ── Guardar dataset_v3 ────────────────────────────────────────────────
PARQUET_V3.parent.mkdir(parents=True, exist_ok=True)
df_v3.to_parquet(PARQUET_V3, index=False)
print(f'Guardado: {PARQUET_V3}')

# Verificar NaNs
nan_cols = df_v3[FEATURE_COLUMNS_V3].isna().sum()
print('\nNaNs por feature:')
print(nan_cols[nan_cols > 0].to_string() or '  Nenhum NaN — OK')


## 3. EDA — Distribuição das novas features de Momentum

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

features_to_plot = MOMENTUM_FEATURES + ['return_5d', 'return_10d', 'return_20d', 'volatility_20d']

for i, feat in enumerate(features_to_plot):
    if feat in df_v3.columns:
        axes[i].hist(df_v3[feat].dropna(), bins=50, edgecolor='none', alpha=0.75, color='steelblue')
        axes[i].axvline(0, color='red', linestyle='--', linewidth=0.8)
        axes[i].set_title(feat, fontsize=10)
        axes[i].set_xlabel('Valor')

plt.suptitle('Distribuição das features de momentum e preço', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Correlação das novas features com os targets
print('Correlações (Spearman) com max_return_60d:')
for feat in FEATURE_COLUMNS_V3:
    if feat in df_v3.columns:
        r, p = spearmanr(df_v3[feat].fillna(0), df_v3['max_return_60d'])
        star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        print(f'  {feat:30s}: rho={r:+.4f}  p={p:.4f}  {star}')


## 4. Walk-Forward Cross-Validation

Janela expansiva (expanding window):  
- **Fold 1**: treino jan2022–jun2023, teste jul2023–dez2023  
- **Fold 2**: treino jan2022–dez2023, teste jan2024–jun2024  
- ... etc

Métricas por fold: Spearman `rho` (rank correlation da pred vs real), Top-K P&L (lucro médio dos alertas no quintil superior do score).

In [ ]:
# ── Helpers de avaliação ─────────────────────────────────────────────

def winsorize(x: np.ndarray, pct: float = 0.01) -> np.ndarray:
    lo, hi = np.percentile(x, pct * 100), np.percentile(x, (1 - pct) * 100)
    return np.clip(x, lo, hi)


def spearman_updown(y_up_pred, y_up_true, y_down_pred, y_down_true) -> dict:
    """Spearman rho para upside e downside separadamente."""    r_up,   _ = spearmanr(y_up_pred,   y_up_true)
    r_down, _ = spearmanr(y_down_pred, y_down_true)
    return {'rho_up': r_up, 'rho_down': r_down, 'rho_mean': (r_up + r_down) / 2}


def topk_pnl(y_up_pred, y_down_pred, y_up_true, k: float = 0.20) -> float:
    """
    Top-K P&L: selecionar os alertas com score mais alto (score = pred_up / abs(pred_down))
    e calcular o max_return_60d médio real nesses alertas.
    Proxy de quanto ganharíamos se seguíssemos o modelo.
    """
    abs_down = np.abs(y_down_pred)
    score    = np.where(abs_down > 0, y_up_pred / abs_down, 0.0)
    n_top    = max(1, int(len(score) * k))
    top_idx  = np.argsort(score)[-n_top:]
    return float(np.mean(y_up_true[top_idx]))


def make_xgb_up():
    return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.05, reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)

def make_xgb_down():
    return XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.025,
        subsample=0.7, colsample_bytree=0.7, min_child_weight=6,
        gamma=0.2, reg_alpha=0.2, reg_lambda=1.5,
        objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)

def make_lgbm_up():
    return lgb.LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.025,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1)

def make_lgbm_down():
    return lgb.LGBMRegressor(n_estimators=600, max_depth=4, learning_rate=0.02,
        subsample=0.7, colsample_bytree=0.7, min_child_samples=30,
        reg_alpha=0.2, reg_lambda=1.5, random_state=42, n_jobs=-1, verbose=-1)

def make_rf_up():
    return RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=10,
        max_features=0.6, random_state=42, n_jobs=-1)

def make_rf_down():
    return RandomForestRegressor(n_estimators=400, max_depth=7, min_samples_leaf=15,
        max_features=0.5, random_state=42, n_jobs=-1)


MODEL_CONFIGS = {
    'XGB-v2':   {'feat': FEATURE_COLUMNS_V2, 'up': make_xgb_up,   'down': make_xgb_down},
    'XGB-v3':   {'feat': FEATURE_COLUMNS_V3, 'up': make_xgb_up,   'down': make_xgb_down},
    'LGBM-v2':  {'feat': FEATURE_COLUMNS_V2, 'up': make_lgbm_up,  'down': make_lgbm_down},
    'LGBM-v3':  {'feat': FEATURE_COLUMNS_V3, 'up': make_lgbm_up,  'down': make_lgbm_down},
    'RF-v2':    {'feat': FEATURE_COLUMNS_V2, 'up': make_rf_up,    'down': make_rf_down},
    'RF-v3':    {'feat': FEATURE_COLUMNS_V3, 'up': make_rf_up,    'down': make_rf_down},
}

print('Configurações de modelos a comparar:')
for name, cfg in MODEL_CONFIGS.items():
    print(f'  {name}: {len(cfg["feat"])} features')


In [ ]:
# ── Walk-Forward CV ────────────────────────────────────────────────────

# Ordenar por data
df_v3 = df_v3.sort_values('alert_date').reset_index(drop=True)
df_v3['alert_date'] = pd.to_datetime(df_v3['alert_date'])

# Definir folds temporais (expanding window)
# Fold i: treino = tudo até corte_i, teste = corte_i até corte_{i+1}
min_date  = df_v3['alert_date'].min()
max_date  = df_v3['alert_date'].max()
total_days = (max_date - min_date).days

N_FOLDS = 5
# Cada fold de teste ocupa 1/N_FOLDS do periodo total
fold_days = total_days // N_FOLDS

# Garantir que treino minimo = 60% dos dados
train_end_dates = [
    min_date + pd.Timedelta(days=int(total_days * (0.6 + 0.08 * k)))
    for k in range(N_FOLDS)
]
test_end_dates = [
    min_date + pd.Timedelta(days=int(total_days * (0.6 + 0.08 * (k + 1))))
    for k in range(N_FOLDS)
]
test_end_dates[-1] = max_date  # garantir cobertura total

print('Walk-Forward folds:')
for k in range(N_FOLDS):
    tr = df_v3[df_v3['alert_date'] <= train_end_dates[k]]
    te = df_v3[(df_v3['alert_date'] > train_end_dates[k]) & (df_v3['alert_date'] <= test_end_dates[k])]
    print(f'  Fold {k+1}: treino {len(tr):5d} amostras  |  teste {len(te):4d} amostras  |  '          f'teste: {train_end_dates[k].date()} → {test_end_dates[k].date()}')


In [ ]:
# ── Correr todos os modelos nos 5 folds ────────────────────────────────
# Esta célula demora ~5-15 min consoante o tamanho do dataset

results: dict[str, list[dict]] = {name: [] for name in MODEL_CONFIGS}

for k in range(N_FOLDS):
    tr_mask = df_v3['alert_date'] <= train_end_dates[k]
    te_mask = (df_v3['alert_date'] > train_end_dates[k]) & (df_v3['alert_date'] <= test_end_dates[k])

    df_tr = df_v3[tr_mask].copy()
    df_te = df_v3[te_mask].copy()

    if len(df_tr) < 50 or len(df_te) < 10:
        print(f'Fold {k+1}: insuficiente — a saltar')
        continue

    # Targets do treino
    y_up_tr   = winsorize(df_tr['max_return_60d'].fillna(0).values)
    y_down_tr = winsorize(df_tr['max_drawdown_60d'].fillna(0).values)
    y_up_tr_t, y_down_tr_t = transform_targets(y_up_tr, y_down_tr)

    # Targets do teste (espaço original, nao transformado — para medir real)
    y_up_te   = df_te['max_return_60d'].fillna(0).values
    y_down_te = df_te['max_drawdown_60d'].fillna(0).values

    for model_name, cfg in MODEL_CONFIGS.items():
        feats = [c for c in cfg['feat'] if c in df_tr.columns]
        X_tr = df_tr[feats].fillna(0).values.astype(np.float32)
        X_te = df_te[feats].fillna(0).values.astype(np.float32)

        m_up   = cfg['up']()
        m_down = cfg['down']()
        m_up.fit(X_tr,   y_up_tr_t)
        m_down.fit(X_tr, y_down_tr_t)

        up_pred_t   = m_up.predict(X_te.astype(np.float32))
        down_pred_t = m_down.predict(X_te.astype(np.float32))
        pred_up, pred_down = inverse_transform(up_pred_t, down_pred_t)

        metrics = spearman_updown(pred_up, y_up_te, pred_down, y_down_te)
        pnl     = topk_pnl(pred_up, pred_down, y_up_te, k=0.20)

        results[model_name].append({
            'fold':       k + 1,
            'n_test':     len(df_te),
            'rho_up':     metrics['rho_up'],
            'rho_down':   metrics['rho_down'],
            'rho_mean':   metrics['rho_mean'],
            'topk_pnl':   pnl,
        })

    print(f'  Fold {k+1} OK')

print('\nWalk-Forward CV concluído.')


## 5. Resultados — Comparação de Modelos

In [ ]:
# ── Tabela resumo ─────────────────────────────────────────────────────
summary_rows = []
for model_name, folds in results.items():
    if not folds:
        continue
    df_folds = pd.DataFrame(folds)
    summary_rows.append({
        'Modelo':      model_name,
        'n_features':  len(MODEL_CONFIGS[model_name]['feat']),
        'rho_up':      df_folds['rho_up'].mean(),
        'rho_down':    df_folds['rho_down'].mean(),
        'rho_mean':    df_folds['rho_mean'].mean(),
        'topk_pnl':    df_folds['topk_pnl'].mean(),
        'rho_std':     df_folds['rho_mean'].std(),
    })

df_summary = pd.DataFrame(summary_rows).sort_values('rho_mean', ascending=False)
print('\n=== Resultado Walk-Forward (média dos 5 folds) ===')
print(df_summary.to_string(index=False))


In [ ]:
# ── Gráfico de comparação ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# rho_mean por modelo
colors = ['#2196F3', '#1565C0', '#4CAF50', '#1B5E20', '#FF9800', '#E65100']
models_sorted = df_summary['Modelo'].tolist()
rho_vals      = df_summary['rho_mean'].tolist()
pnl_vals      = df_summary['topk_pnl'].tolist()

bars1 = ax1.barh(models_sorted, rho_vals, color=colors[:len(models_sorted)])
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_xlabel('Spearman rho (média folds)')
ax1.set_title('Rank Correlation — upside + downside')
for bar, v in zip(bars1, rho_vals):
    ax1.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

bars2 = ax2.barh(models_sorted, pnl_vals, color=colors[:len(models_sorted)])
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Top-20% avg max_return_60d')
ax2.set_title('Top-K P&L — retorno real nos alertas selecionados')
for bar, v in zip(bars2, pnl_vals):
    ax2.text(v + 0.001, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Comparação de Modelos — Walk-Forward CV (5 folds)', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Evolução por fold (estabilidade ao longo do tempo) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.get_cmap('tab10')
for i, (model_name, folds) in enumerate(results.items()):
    if not folds:
        continue
    df_f = pd.DataFrame(folds).sort_values('fold')
    color = cmap(i)
    axes[0].plot(df_f['fold'], df_f['rho_mean'],  marker='o', label=model_name, color=color)
    axes[1].plot(df_f['fold'], df_f['topk_pnl'],  marker='s', label=model_name, color=color)

for ax, title, ylabel in zip(axes,
    ['Spearman rho — evolução temporal', 'Top-K P&L — evolução temporal'],
    ['rho_mean', 'topk_pnl']):
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Fold')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xticks(range(1, N_FOLDS + 1))

plt.suptitle('Estabilidade dos modelos ao longo do tempo', fontsize=13)
plt.tight_layout()
plt.show()


## 6. Feature Importance — Champion vs Challenger

In [ ]:
# ── Selecção automática do champion ────────────────────────────────────
# Champion = melhor rho_mean médio nos 5 folds
# Desempate: maior topk_pnl

CHAMPION_NAME = df_summary.sort_values(
    ['rho_mean', 'topk_pnl'], ascending=False
).iloc[0]['Modelo']

print(f'Champion: {CHAMPION_NAME}')
print()
print(df_summary.to_string(index=False))


In [ ]:
# ── Treinar champion no dataset COMPLETO ───────────────────────────────

champion_cfg   = MODEL_CONFIGS[CHAMPION_NAME]
champion_feats = [c for c in champion_cfg['feat'] if c in df_v3.columns]

X_full    = df_v3[champion_feats].fillna(0).values.astype(np.float32)
y_up_full = winsorize(df_v3['max_return_60d'].fillna(0).values)
y_dn_full = winsorize(df_v3['max_drawdown_60d'].fillna(0).values)
y_up_t, y_dn_t = transform_targets(y_up_full, y_dn_full)

print(f'A treinar {CHAMPION_NAME} no dataset completo ({len(X_full)} amostras, {len(champion_feats)} features)...')

champ_up   = champion_cfg['up']()
champ_down = champion_cfg['down']()
champ_up.fit(X_full,   y_up_t)
champ_down.fit(X_full, y_dn_t)

print('Treino concluído.')


In [ ]:
# ── Feature importance do champion ────────────────────────────────────

def get_importance(model, feat_names: list[str]) -> pd.Series:
    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
    else:
        imp = np.zeros(len(feat_names))
    return pd.Series(imp, index=feat_names).sort_values(ascending=False)

imp_up   = get_importance(champ_up,   champion_feats)
imp_down = get_importance(champ_down, champion_feats)
imp_mean = (imp_up + imp_down) / 2

# Top 20 features
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, imp, title in zip(axes,
    [imp_up, imp_down, imp_mean],
    ['Upside model (max_return_60d)', 'Downside model (max_drawdown_60d)', 'Média (upside + downside)']):
    top20 = imp.head(20)
    # Colorir features de momentum a verde
    colors = ['#1B5E20' if f in MOMENTUM_FEATURES else '#2196F3' for f in top20.index]
    ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
    ax.set_title(f'{CHAMPION_NAME} — {title}', fontsize=10)
    ax.set_xlabel('Importance')

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2196F3', label='Feature existente'),
                   Patch(facecolor='#1B5E20', label='Feature momentum (nova)')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f'Feature Importance — {CHAMPION_NAME} (champion)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nTop 10 features (média upside+downside):')
print(imp_mean.head(10).to_string())


## 7. Guardar champion — dip_models_v3.pkl

In [ ]:
from experiments.ml_v2.pipeline import DipModelsV2

# Reutilizar o container DipModelsV2 (compatível com predict_v2)
champion_models = DipModelsV2(
    model_up=champ_up,
    model_down=champ_down,
    feature_cols=champion_feats,
    n_train_samples=len(X_full),
    train_date=datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
)

out_pkl = MODELS_DIR / 'dip_models_v3.pkl'
with open(out_pkl, 'wb') as f:
    pickle.dump(champion_models, f)

print(f'Champion guardado: {out_pkl}')
print(f'  Modelo    : {CHAMPION_NAME}')
print(f'  Features  : {len(champion_feats)}')
print(f'  Amostras  : {len(X_full)}')
print(f'  Train date: {champion_models.train_date}')


## 8. ml_report_v3.json — Métricas para o Health Monitor

In [ ]:
import json

champ_row = df_summary[df_summary['Modelo'] == CHAMPION_NAME].iloc[0]

report = {
    'schema_version':  3,
    'trained_at':      datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'champion_model':  CHAMPION_NAME,
    'n_features':      len(champion_feats),
    'n_train_samples': int(len(X_full)),
    'feature_cols':    champion_feats,
    'momentum_features_added': MOMENTUM_FEATURES,
    'walk_forward_cv': {
        'n_folds':        N_FOLDS,
        'rho_up_mean':    float(champ_row['rho_up']),
        'rho_down_mean':  float(champ_row['rho_down']),
        'rho_mean':       float(champ_row['rho_mean']),
        'rho_std':        float(champ_row['rho_std']),
        'topk_pnl_mean':  float(champ_row['topk_pnl']),
    },
    'all_models': df_summary[['Modelo', 'rho_mean', 'topk_pnl']].to_dict(orient='records'),
    'top_features': imp_mean.head(15).to_dict(),
    'dataset_stats': {
        'n_samples':        int(len(df_v3)),
        'win_rate':         float(df_v3['label_win'].mean()),
        'max_return_mean':  float(df_v3['max_return_60d'].mean()),
        'max_return_std':   float(df_v3['max_return_60d'].std()),
        'max_drawdown_mean': float(df_v3['max_drawdown_60d'].mean()),
        'max_drawdown_std':  float(df_v3['max_drawdown_60d'].std()),
    },
}

report_path = REPO_ROOT / 'ml_report_v3.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'ml_report_v3.json guardado: {report_path}')
print()
print(json.dumps(report, indent=2))


## 9. Próximos passos

Após correr este notebook:

1. **Se o champion melhorou** (rho_mean > modelo produção actual):
   - Copiar `dip_models_v3.pkl` para a raiz do repo como `dip_model_stage1.pkl` / `dip_model_stage2.pkl`
   - Ou actualizar `ml_predictor.py` para carregar o novo formato `DipModelsV2`
   - Copiar `ml_report_v3.json` → `ml_report.json`

2. **Actualizar `monthly_retrain.py`** para passar `sector_history` e `spy_history` ao `build_features()`
   (já suportado no novo `ml_features.py` — só falta passar os parâmetros)

3. **Adicionar cron no Railway** para o retreino mensal automático
